# Lab: Demand Estimation and Industrial Organization

## Bluetooth Speakers
- The electronics product that had the nicest characteristics for our lab is: Bluetooth Speakers
- The top 5 brands are Bose, JBL, Anker, DOSS, OontZ, plus the **outside option** of Other, which are a collection of small and cheap brands with typically bad reviews
- We want to look at sales data (prices, choices) and infer whether the market is performing well for consumers: Is it competitive? Are prices reasonable or not, relative to cost?

## Herfindahl-Hirschman Index
- The **market share for brand $i$** is the proportional of total sales that correspond to $i$'s product
- So if $N$ units are sold in the market and $n_i$ are from brand $i$, $s_i = n_i/N$
- Anti-trust authorities like the DOJ and FTC regularly compute the Herfindahl-Hirschman Index as a metric of how concentracted a market is:
$$
HHI = \sum_{i=1}^N s_i^2
$$
Plot the HHI over time for bluetooth speakers
- Anti-trust authorities consider an HHI < 1500 to be competitive, 1500 to 2500 to be moderately concentrated, and above 2500 to be highly concentrated. Is this market relatively competitive, or not?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LinearRegression

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LinearRegression

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


In [ ]:
consumer_df = pd.read_parquet("demand_1.parquet", engine="fastparquet").copy()
consumer_df.head()


In [ ]:
consumer_summary = (
    consumer_df.groupby("choice", as_index=False)
    .agg(
        purchases=("user_id", "count"),
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        avg_rating=("average_rating", "mean"),
    )
    .sort_values("purchases", ascending=False)
)

consumer_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.kdeplot(
    data=consumer_df,
    x="price",
    hue="choice",
    common_norm=False,
    ax=axes[0],
)
axes[0].set_title("Price Distribution by Brand")

sns.kdeplot(
    data=consumer_df,
    x="average_rating",
    hue="choice",
    common_norm=False,
    ax=axes[1],
)
axes[1].set_title("Average Rating Distribution by Brand")

plt.tight_layout()
plt.show()


## 1. Consumer EDA

Load `demand_1.parquet`. This is consumer level data, which is already substantially cleaned and organized from the intial Amazon review dump.

- Make kernel density plots for the price and average rating of the products, by brand, as well as describe tables grouped by brand. Which are most expensive? Which are most popular?

In [ ]:
brand_df = pd.read_parquet("demand_2.parquet", engine="fastparquet").copy()
brand_df.head()


In [ ]:
sales_pivot = (
    brand_df.pivot(index="year", columns="choice", values="chosen_users")
    .fillna(0)
)

price_pivot = brand_df.pivot(index="year", columns="choice", values="avg_price")

share_pivot = (
    brand_df.pivot(index="year", columns="choice", values="share_among_speaker_buyers")
    .fillna(0)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sales_pivot.plot(marker="o", ax=axes[0])
axes[0].set_title("Brand Sales by Year")
axes[0].set_ylabel("Chosen Users")

price_pivot.plot(marker="o", ax=axes[1])
axes[1].set_title("Average Price by Brand and Year")
axes[1].set_ylabel("Price ($)")

share_pivot.plot(marker="o", ax=axes[2])
axes[2].set_title("Market Share by Brand and Year")
axes[2].set_ylabel("Share")

plt.tight_layout()
plt.show()


In [ ]:
hhi_df = (
    brand_df.groupby("year", as_index=False)
    .agg(hhi=("share_among_speaker_buyers", lambda s: 10000 * np.square(s).sum()))
)

hhi_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=hhi_df, x="year", y="hhi", marker="o", ax=ax)
ax.axhline(1500, color="orange", linestyle="--", label="Moderately concentrated")
ax.axhline(2500, color="red", linestyle="--", label="Highly concentrated")
ax.set_title("HHI Over Time")
ax.set_ylabel("HHI")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
consumer_df = pd.read_parquet("demand_1.parquet", engine="fastparquet").copy()
consumer_df.head()


In [ ]:
consumer_summary = (
    consumer_df.groupby("choice", as_index=False)
    .agg(
        purchases=("user_id", "count"),
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        avg_rating=("average_rating", "mean"),
    )
    .sort_values("purchases", ascending=False)
)

consumer_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.kdeplot(
    data=consumer_df,
    x="price",
    hue="choice",
    common_norm=False,
    ax=axes[0],
)
axes[0].set_title("Price Distribution by Brand")

sns.kdeplot(
    data=consumer_df,
    x="average_rating",
    hue="choice",
    common_norm=False,
    ax=axes[1],
)
axes[1].set_title("Average Rating Distribution by Brand")

plt.tight_layout()
plt.show()


## 2. Brand EDA

Load `demand_2.parquet`. This is brand-level data, which is already consolidated from the consumer level data.

- Make plots of the sales and average prices of each brand in each year
- Plot the market shares of each brand in each year
- Compute the HHI of the market over time, and plot it

In [ ]:
brand_df = pd.read_parquet("demand_2.parquet", engine="fastparquet").copy()
brand_df.head()


In [ ]:
sales_pivot = (
    brand_df.pivot(index="year", columns="choice", values="chosen_users")
    .fillna(0)
)

price_pivot = brand_df.pivot(index="year", columns="choice", values="avg_price")

share_pivot = (
    brand_df.pivot(index="year", columns="choice", values="share_among_speaker_buyers")
    .fillna(0)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sales_pivot.plot(marker="o", ax=axes[0])
axes[0].set_title("Brand Sales by Year")
axes[0].set_ylabel("Chosen Users")

price_pivot.plot(marker="o", ax=axes[1])
axes[1].set_title("Average Price by Brand and Year")
axes[1].set_ylabel("Price ($)")

share_pivot.plot(marker="o", ax=axes[2])
axes[2].set_title("Market Share by Brand and Year")
axes[2].set_ylabel("Share")

plt.tight_layout()
plt.show()


In [ ]:
demand_df = pd.read_parquet("demand_3.parquet", engine="fastparquet").copy()

feature_cols = [
    "year",
    "choice",
    "waterproof_share",
    "party_share",
    "voice_assistant_share",
]

feature_df = brand_df[feature_cols].copy()
outside_df = (
    brand_df.loc[brand_df["choice"] == "Other", ["year", "share_among_speaker_buyers"]]
    .rename(columns={"share_among_speaker_buyers": "outside_share"})
)

demand_df = demand_df.merge(feature_df, on=["year", "choice"], how="left")
demand_df = demand_df.merge(outside_df, on="year", how="left")
demand_df["log_share_diff"] = (
    np.log(demand_df["share_among_speaker_buyers"].clip(lower=1e-12))
    - np.log(demand_df["outside_share"].clip(lower=1e-12))
)

demand_df.head()


In [ ]:
X = demand_df[
    [
        "avg_price",
        "avg_rating",
        "waterproof_share",
        "party_share",
        "voice_assistant_share",
    ]
].copy()

X = pd.concat(
    [
        X,
        pd.get_dummies(demand_df["choice"], prefix="choice", drop_first=True),
        pd.get_dummies(demand_df["year"], prefix="year", drop_first=True),
    ],
    axis=1,
)

y = demand_df["log_share_diff"].copy()

model = LinearRegression()
model.fit(X, y)

coef = pd.Series(model.coef_, index=X.columns).sort_values()
coef


In [ ]:
price_coef = float(pd.Series(model.coef_, index=X.columns)["avg_price"])
price_coef


In [ ]:
hhi_df = (
    brand_df.groupby("year", as_index=False)
    .agg(hhi=("share_among_speaker_buyers", lambda s: 10000 * np.square(s).sum()))
)

hhi_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=hhi_df, x="year", y="hhi", marker="o", ax=ax)
ax.axhline(1500, color="orange", linestyle="--", label="Moderately concentrated")
ax.axhline(2500, color="red", linestyle="--", label="Highly concentrated")
ax.set_title("HHI Over Time")
ax.set_ylabel("HHI")
ax.legend()
plt.tight_layout()
plt.show()


## Demand
- We typically think that product characteristics like price and features determines which product consumers purchase
- We don't observe the benefit or utility that corresponds to how much consumers like products, just the product they ultimately purchase; this is the latent variable being modeled
- This lends itself really well to multinomial logistic regression (but with a twist)
- This kind of model appears in economics in **industrial organization** and in business as **quantitative marketing**

## Demand Systems
- Imagine consumer $i$ deciding between different brands $j=1, 2, ... , J$ in period $t$, with prices $p_{j}$ and characteristics $x_{jt}$
- The **mean utility** provided by product $j$ to consumer $i$ is
$$
\delta_{jt} = b_0  + \sum_{\ell =1}^L b_\ell x_{jt\ell}- b_{\text{price}} p_{jt}
$$
- So consumers get an average benefit that depends on product characteristics, and then the price reduces that benefit
- When we add an error term and assume that it has the logistic distribution, this becomes multinomial logistic regression: Consumer $i$ gets a shock $\varepsilon_{ijt}$, and gets utilities for each brand $j$ of
$$
U_{ijt} = \delta_{ijt} + \varepsilon_{ijt} = b_0  + \sum_{\ell =1}^L b_\ell x_{jt\ell}- b_{\text{price}} p_{jt}+ \varepsilon_{ijt}
$$

In [ ]:
market_df = pd.read_parquet("demand_4.parquet", engine="fastparquet").copy()

market_df["markup"] = -market_df["demand_users"] / market_df["dprime"]
market_df["marginal_cost"] = market_df["avg_price"] - market_df["markup"]
market_df["profit"] = market_df["markup"] * market_df["demand_users"]

market_df.head()


In [ ]:
market_df[["markup", "marginal_cost", "profit"]].describe().round(2)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.kdeplot(data=market_df, x="marginal_cost", hue="choice", common_norm=False, ax=axes[0])
axes[0].set_title("Unit Cost Distribution")

sns.kdeplot(data=market_df, x="markup", hue="choice", common_norm=False, ax=axes[1])
axes[1].set_title("Markup Distribution")

sns.kdeplot(data=market_df, x="profit", hue="choice", common_norm=False, ax=axes[2])
axes[2].set_title("Profit Distribution")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=market_df, x="avg_price", y="marginal_cost", hue="choice", ax=axes[0])
axes[0].set_title("Price vs Marginal Cost")

sns.scatterplot(data=market_df, x="avg_rating", y="marginal_cost", hue="choice", ax=axes[1])
axes[1].set_title("Average Rating vs Marginal Cost")

plt.tight_layout()
plt.show()


In [ ]:
market_df.groupby("choice", as_index=False)[["markup", "marginal_cost", "profit"]].mean().round(2)


## Demand Systems
- The product with the highest $U_{ijt}$ is the one they purchase, but market shares are determined by
$$
s_{jt} = \begin{cases} 
\frac{\exp(\delta_{jt})}{1 + \sum_{k=1}^{K-1} \exp(\delta_{kt})}, & k = 1, ..., K \\
\frac{1}{1 + \sum_{k=1}^{K-1}\exp(\delta_{kt})}, & \text{outside option}
\end{cases}
$$
- This is great: We just collect product and consumption data, and we can infer demand
- You *could* run this on the original consumer dataset with `.LogisticRegression()`, but that would be very unstable and take forever to converge

## Log-Linearizing the Demand System
- It turns out that estimating demand systems from millions of customers is extremely unstable
- Instead, we'll use a trick to reduce multinomial logistic regression to linear regression
- For the outside option, set $\delta_{Kt} = 0$; this is allowed, because we don't know the scale of the unobserved latent variable anyway -- we're just normalizing the "units" of the unobserved latent scale
- Now, notice that if we divide the share of any brand by demand for the outside option, we get
$$
\dfrac{s_{jt}}{s_{Kt}} = \frac{\exp(\delta_{jt})}{\exp(\delta_{Kt})}
$$
- Now take logs:
$$
\log(s_{jt}) - \log(s_{Kt}) = \delta_{jt} - \underbrace{\delta_{Kt}}_{=0}
$$
- And finally,
$$
\underbrace{\log(s_{jt}) - \log(s_{Kt})}_{\text{Difference in Log Shares}} = \underbrace{b_0  + \sum_{\ell =1}^L b_\ell x_{jt\ell}- b_{\text{price}} p_{jt}}_{\text{Linear Model}}
$$


## Log-Linearizing the Demand System
- So, we can convert from consumer-level data to brand-level data, and estimate demand using product shares instead of individual consumer choices
- The regression we want to run is:
$$
\underbrace{\log(s_{jt}) - \log(s_{Kt})}_{\text{Difference in Log Shares}} = \underbrace{b_0  + \sum_{\ell =1}^L b_\ell x_{jt\ell}- b_{\text{price}} p_{jt}}_{\text{Linear Model}}
$$
- So we just compute the log share differences on the left, and use it as our usual $y_{jt}$, and feature engineer the right-hand side
- Learning about consumer preferences and product positioning is **quantitative marketing**

## 3. Estimate the Demand System

Load `demand_3.parquet`. This is brand-level data, which is already consolidated from the consumer level data and aggregated across products sold by that brand.

- The log share differences are already computed, as `log_share_diff`
- Regress those values on price, waterproof flag, party flag, voice assistant flag, year dummies, average rating, and brand dummies

1. What is the coefficient on price? Is it negative? If the price is/were positive, does that make sense? Why might that happen?
2. Which brands have the highest coefficients?
3. Which features (waterproof, voice assistant, party) increase market share, which seem to decrease it?

In [ ]:
demand_df = pd.read_parquet("demand_3.parquet", engine="fastparquet").copy()

feature_cols = [
    "year",
    "choice",
    "waterproof_share",
    "party_share",
    "voice_assistant_share",
]

feature_df = brand_df[feature_cols].copy()
outside_df = (
    brand_df.loc[brand_df["choice"] == "Other", ["year", "share_among_speaker_buyers"]]
    .rename(columns={"share_among_speaker_buyers": "outside_share"})
)

demand_df = demand_df.merge(feature_df, on=["year", "choice"], how="left")
demand_df = demand_df.merge(outside_df, on="year", how="left")
demand_df["log_share_diff"] = (
    np.log(demand_df["share_among_speaker_buyers"].clip(lower=1e-12))
    - np.log(demand_df["outside_share"].clip(lower=1e-12))
)

demand_df.head()


In [ ]:
X = demand_df[
    [
        "avg_price",
        "avg_rating",
        "waterproof_share",
        "party_share",
        "voice_assistant_share",
    ]
].copy()

X = pd.concat(
    [
        X,
        pd.get_dummies(demand_df["choice"], prefix="choice", drop_first=True),
        pd.get_dummies(demand_df["year"], prefix="year", drop_first=True),
    ],
    axis=1,
)

y = demand_df["log_share_diff"].copy()

model = LinearRegression()
model.fit(X, y)

coef = pd.Series(model.coef_, index=X.columns).sort_values()
coef


In [ ]:
price_coef = float(pd.Series(model.coef_, index=X.columns)["avg_price"])
price_coef


## Estimating Unit/Marginal Costs
- These firms pick their prices strategically, and price typically exceeds the true cost
- We want to understand how much the firms are making on each unit, in excess of the cost of creating it
- This is where economics begins: Do our markets maximize welfare? How much money are firms making by pricing strategically above cost? Are consumers exploited or not?
- The Amazon data are nice for data science purposes, but the same kind of analysis applies to education or epinephrine

## Inferring Cost
- OK, buckle up
- Firm $j$ in period $t$ is maximizing its profits:
$$
\pi_{jt} = \underbrace{s_{jt}(p_{jt})}_{\text{Quantity Sold}} \quad \underbrace{(p_{jt}-c_{jt})}_{\text{Profit per Unit}}
$$
- That means that, at a maximum, it must be true that
$$
\underbrace{s_{jt}'(p_{jt})(p_{jt}-c_{jt})}_{\text{Lost sales on the margin}} + \underbrace{s_{jt}(p_{jt}-c_{jt})}_{\text{Profits on units sold}} = 0
$$
- If we re-arrange to solve for $c_{jt}$, we get
$$
\hat{c}_{jt} = p_{jt} + \dfrac{s_{jt}(p_{jt})}{s_{jt}'(p_{jt})}
$$
- We can estimate the firm's unobserved, true cost of its product from demand behavior and prices
- Notice that price always exceeds marginal cost, here: Brands always price in excess of true cost

## Slope of Demand
- To keep things humane, I've pre-computed the derivative of the demand curve, like this:
$$
s_{jt}'(p_{jt}) = \frac{s_{jt}(p_{jt}+1)-s_{jt}(p_{jt}-1)}{2}
$$

## 4. Market Fundamentals

Load `demand_4.parquet`. I've already computed $s_{jt}'(p_{jt})$ as `dprime`

- Compute the **unit cost** $\hat{c}_{jt}$ for each Brand in each period
- Compute the **mark-up** in each period, $markup_{jt} = p_{jt} - \hat{c}_{jt}$
- Compute the brand's **average profit** in each period, $ s_{jt} \times markup_{jt}$
- Make a kernel density plot of unit costs, mark-ups, and profits for each brand-year
- Which brands are most profitable? Which have the highest unit costs?
- Make scatter plots of price and unit cost, and average review and unit cost. Do you notice any patterns?

In [ ]:
market_df = pd.read_parquet("demand_4.parquet", engine="fastparquet").copy()

market_df["markup"] = -market_df["demand_users"] / market_df["dprime"]
market_df["marginal_cost"] = market_df["avg_price"] - market_df["markup"]
market_df["profit"] = market_df["markup"] * market_df["demand_users"]

market_df.head()


In [ ]:
market_df[["markup", "marginal_cost", "profit"]].describe().round(2)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.kdeplot(data=market_df, x="marginal_cost", hue="choice", common_norm=False, ax=axes[0])
axes[0].set_title("Unit Cost Distribution")

sns.kdeplot(data=market_df, x="markup", hue="choice", common_norm=False, ax=axes[1])
axes[1].set_title("Markup Distribution")

sns.kdeplot(data=market_df, x="profit", hue="choice", common_norm=False, ax=axes[2])
axes[2].set_title("Profit Distribution")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=market_df, x="avg_price", y="marginal_cost", hue="choice", ax=axes[0])
axes[0].set_title("Price vs Marginal Cost")

sns.scatterplot(data=market_df, x="avg_rating", y="marginal_cost", hue="choice", ax=axes[1])
axes[1].set_title("Average Rating vs Marginal Cost")

plt.tight_layout()
plt.show()


In [ ]:
market_df.groupby("choice", as_index=False)[["markup", "marginal_cost", "profit"]].mean().round(2)


## 5. Wrapping Up
The DOJ and FTC use these kinds of models for anti-trust investigations. Online commerce is the target of recent FTC regulatory actions, in particular towards Amazon. 

1. Do you think this market is relatively competitive?
2. Are the mark-ups sizeable? Are they worth pursuing regulatory or industrial policy over, or even an anti-trust lawsuit?
3. The bigger picture question is: "What is the market definition, and what products should be included?" The broader the market defintion, the more finely demand and sales are split, and the less power every firm appears to have (e.g. baseball; professional sports; entertainment). If you had to defend the brands we analyzed, what argument would you make?